# Test Pasos 20-27: Tabla Final de Inversiones

> **Propósito**: Notebook limpio para probar los pasos 20-27 del modelo de inversiones.
> 
> **Prerequisitos**: Los flujos de los pasos 1-19 ya deben estar calculados.
> 
> **Fecha**: 2026-02-05

## 1. Setup y Carga de Dependencias

In [ ]:
import pandas as pd
import numpy as np
import pickle
import sys
from pathlib import Path
from datetime import datetime

# Configuración de paths
notebook_path = Path().resolve()
BASE_DIR = notebook_path
while not (BASE_DIR / '.git').exists():
    BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# Path a datos externos
PROCESOS_DIARIOS_PATH = BASE_DIR.parent / 'PROCESOS_DIARIOS_MODELOS'
DATA_PATH = PROCESOS_DIARIOS_PATH / 'data' / 'external' / 'ml_inversiones'

# Fecha de proceso
FECHA_PROCESO = 20260115

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_PATH: {DATA_PATH}")
print(f"FECHA_PROCESO: {FECHA_PROCESO}")

## 2. Importar Módulos del Proyecto

In [ ]:
# Importar módulo de tabla final (pasos 20-27)
from RF_Modelo_Inversiones.output.tabla_final import (
    generar_precios_dia,
    formatear_flujo_instrumento,
    generar_tabla_final_inversiones,
    agregar_precio_y_flujo_clp,
    generar_tabla_desarrollo_completa,
    formatear_para_excel,
    ejecutar_pasos_20_a_27,
    COLUMNAS_TABLA_FINAL,
    COLUMNAS_TABLA_DESARROLLO,
    MAPEO_COLUMNAS_EXCEL
)

# Importar orquestador (para generar flujos si es necesario)
from RF_Modelo_Inversiones.pipeline.orquestador import (
    generar_flujo_liquidacion_instrumento,
    listar_tipos_instrumento
)

# Importar funciones de cartera
from RF_Modelo_Inversiones.pipeline.cartera import (
    genera_cartera_inv
)

print("✅ Módulos importados correctamente")
print(f"   - Columnas tabla final: {len(COLUMNAS_TABLA_FINAL)}")
print(f"   - Columnas tabla desarrollo: {len(COLUMNAS_TABLA_DESARROLLO)}")
print(f"   - Instrumentos disponibles: {listar_tipos_instrumento()}")

## 3. Cargar Datos Base (Tablas Linkeadas)

Usamos `cargar_tablas_ml_inversiones()` que:
- Abstrae la fuente de datos (PICKLE/LIVE/BIGQUERY)
- Aplica transformaciones automáticas (FPL, RF_MontosLiq)
- Permite cambiar de modo con variable de entorno o parámetro

In [4]:
# Usar el nuevo sistema de carga de datos
from RF_Modelo_Inversiones.io import (
    cargar_tablas_ml_inversiones, 
    DataSourceMode,
    listar_transformaciones
)

# Cargar tablas (modo PICKLE por defecto, con transformaciones automáticas)
tablas = cargar_tablas_ml_inversiones(
    fecha_proceso=FECHA_PROCESO,
    data_path=DATA_PATH,
    modo=DataSourceMode.PICKLE,
    verbose=True
)

print(f"\n📊 Tablas cargadas ({len(tablas)}):")
for nombre, df in tablas.items():
    print(f"   - {nombre}: {df.shape[0]:,} filas x {df.shape[1]} cols")

# Mostrar transformaciones aplicadas
print(f"\n🔧 Transformaciones registradas:")
for tabla, doc in listar_transformaciones().items():
    print(f"   - {tabla}: {doc.split(chr(10))[0]}")

📊 Cargando tablas ML Inversiones (modo: pickle)
  ✓ Cargado desde pickle: tablas_linkeadas_ml_inversiones_20260115_20260119_101906.pkl
  ✓ 12 tablas cargadas

📊 Tablas cargadas (12):
   - FPL: 7 filas x 2 cols
   - RF_base_Completa_Hist_Input: 414,802 filas x 44 cols
   - RF_Base_Diaria_Precios: 455,659 filas x 24 cols
   - RF_Base_Diaria_Precios_: 244,010 filas x 24 cols
   - RF_BD_Gestion_RM: 146,955 filas x 23 cols
   - RF_Cartera_RtaFija_Hist: 218,556 filas x 20 cols
   - RF_FactCLF_Banc: 1,350 filas x 4 cols
   - RF_FactCLF_Gob: 1,350 filas x 4 cols
   - RF_FactCLP_Banc: 1,260 filas x 4 cols
   - RF_FactCLP_Gob: 1,260 filas x 4 cols
   - RF_Fecha_Proceso_Carteras: 1 filas x 1 cols
   - RF_MontosLiq: 7 filas x 4 cols

🔧 Transformaciones registradas:
   - FPL: 
   - RF_MontosLiq: 


In [5]:
# Cargar tablas de Access (para validación)
archivos_access = list(DATA_PATH.glob(f"tablas_access_prod_{FECHA_PROCESO}_*.pkl"))
if archivos_access:
    archivo_access = max(archivos_access, key=lambda x: x.stat().st_mtime)
    print(f"📦 Cargando tablas Access: {archivo_access.name}")
    with open(archivo_access, "rb") as f:
        tablas_access = pickle.load(f)
    print(f"   Tablas disponibles: {list(tablas_access.keys())}")
else:
    print("⚠️ No hay tablas de Access para validación")
    tablas_access = {}

📦 Cargando tablas Access: tablas_access_prod_20260115_20260119_102556.pkl
   Tablas disponibles: ['Flujo_BBC', 'Flujo_DPF', 'Flujo_DPR', 'Flujo_GobCLF', 'Flujo_GobCLP', 'Flujo_LCH', 'Precios_Dia', 'RF_base_Completa_Hist', 'RF_base_Completa_Hist_30-09', 'RF_BD_Gestion_RL_30-09-2021', 'RF_CarteraBBC_Pond', 'RF_CarteraDPF_Pond', 'RF_CarteraDPR_Pond', 'RF_CarteraGobCLF_Pond', 'RF_CarteraGobCLP_Pond', 'RF_CarteraLCH_Pond', 'RF_Fecha_Proceso_Carteras_', 'RF_LI_MODELO_SALIDA_CONT_DERIV', 'RF_PLI_Modelo_Inversiones_Final_CLP', 'RF_Tabla_Desarrollo_Final', 'RF_Tabla_Desarrollo_Interna']


## 4. Preparar Tabla Base y Generar Flujos (Pasos 1-19)

Si los flujos no están pre-calculados, los generamos aquí.

In [6]:
# Importar helpers para generar RF_base_Completa_Hist
import importlib
import RF_Modelo_Inversiones.dev.helpers as helpers
importlib.reload(helpers)

# Generar tabla base histórica filtrada por fecha
tablas['RF_base_Completa_Hist'] = helpers.genera_tabla_RF_base_Completa_Hist(
    tablas['RF_base_Completa_Hist_Input'], 
    FECHA_PROCESO
)
print(f"✅ RF_base_Completa_Hist: {len(tablas['RF_base_Completa_Hist']):,} filas")

✅ RF_base_Completa_Hist: 414,802 filas


In [7]:
# Generar cartera de inversiones y pactos
tabla_fecha = pd.DataFrame({'Fecha': [pd.to_datetime(str(FECHA_PROCESO), format="%Y%m%d")]})

# Cartera disponible (pasos 1-19)
df_cartera_inv = genera_cartera_inv(
    df_base=tablas['RF_base_Completa_Hist'],
    df_fecha=tabla_fecha,
    tipo='disponible',
    verbose=True
)

# Cartera pactos (para pactos >90 días)
df_cartera_pacto = genera_cartera_inv(
    df_base=tablas['RF_base_Completa_Hist'],
    df_fecha=tabla_fecha,
    tipo='pacto',
    verbose=True
)

print(f"\n📊 Resumen:")
print(f"   - Cartera disponible: {len(df_cartera_inv):,} registros")
print(f"   - Cartera pactos: {len(df_cartera_pacto):,} registros")


GENERANDO CARTERA: Cartera Inversiones Disponible (RF_PLI_001_CarteraInv)
Registros entrada: 414,802

[JOIN] Filtro fecha proceso = 2026-01-15
  Registros que cumplen: 51,748

[WHERE] Excluir LCH: Nemotecnico[:3] <> 'LCH'
  Registros que cumplen: 408,778

[WHERE] Cod_Pro es inversión financiera
  Registros que cumplen: 11,718

[WHERE] Cod_Sub_Pro termina en: Disp, Disp_Liq, MUTUOS
  Registros que cumplen: 11,493

[WHERE] Excluir HTM: Clasificacion_Contable <> 'HTM'
  Registros que cumplen: 408,578

[WHERE FINAL] Todos los filtros combinados (AND)
  Registros que cumplen: 675

RESULTADO: 675 registros

  Distribución Instrumento:
Instrumento
BBC    356
BTP    232
BTU     68
PDB      6
DPF      6
DPR      5
OTR      2

GENERANDO CARTERA: Cartera Inversiones Pacto (RF_PLI_001d_CarteraInv_Pcto)
Registros entrada: 414,802

[JOIN] Filtro fecha proceso = 2026-01-15
  Registros que cumplen: 51,748

[WHERE] Cod_Pro es inversión financiera
  Registros que cumplen: 11,718

[WHERE] Cod_Sub_Pro te

In [8]:
# Generar flujos de liquidación para los 6 instrumentos
INSTRUMENTOS = ['GobCLP', 'GobCLF', 'DPF', 'DPR', 'BBC', 'LCH']
flujos = {}
queries_por_instrumento = {}

for instrumento in INSTRUMENTOS:
    print(f"\n{'='*60}")
    print(f"Procesando: {instrumento}")
    print('='*60)
    
    # Determinar tabla de factores
    if instrumento in ['GobCLP']:
        tabla_factores = 'RF_FactCLP_Gob'
    elif instrumento in ['GobCLF']:
        tabla_factores = 'RF_FactCLF_Gob'
    elif instrumento in ['DPF', 'BBC']:
        tabla_factores = 'RF_FactCLP_Banc'
    else:  # DPR, LCH
        tabla_factores = 'RF_FactCLF_Banc'
    
    tablas_inst = {
        tabla_factores: tablas[tabla_factores],
        'FPL': tablas['FPL'],
        'RF_MontosLiq': tablas['RF_MontosLiq']
    }
    
    flujo, queries_inst = generar_flujo_liquidacion_instrumento(
        df_cartera_inv=df_cartera_inv,
        df_cartera_inv_pacto=df_cartera_pacto,
        tablas=tablas_inst,
        tipo_instrumento=instrumento,
        fecha_proceso=FECHA_PROCESO,
        verbose=False  # Silenciar para velocidad
    )
    
    flujos[instrumento] = flujo
    queries_por_instrumento[instrumento] = queries_inst
    
    # Resumen
    monto_total = flujo['Monto_Liquidar'].sum() if 'Monto_Liquidar' in flujo.columns else 0
    print(f"   ✅ {instrumento}: {len(flujo)} días, Monto total: {monto_total:,.0f}")

print("\n" + "="*60)
print("✅ FLUJOS GENERADOS (Pasos 1-19 completados)")
print("="*60)


Procesando: GobCLP
   ✅ GobCLP: 91 días, Monto total: 1,250,345,640,998

Procesando: GobCLF
   ✅ GobCLF: 91 días, Monto total: 2,178,726

Procesando: DPF
   ✅ DPF: 91 días, Monto total: 52,122,342,432

Procesando: DPR
   ✅ DPR: 91 días, Monto total: 1,051,090

Procesando: BBC
   ✅ BBC: 91 días, Monto total: 276,303,818,362

Procesando: LCH
   ✅ LCH: 91 días, Monto total: 3,157,700

✅ FLUJOS GENERADOS (Pasos 1-19 completados)


In [9]:
DATA_PATH

WindowsPath('C:/Users/vlandaetat/code/PROCESOS_DIARIOS_MODELOS/data/external/ml_inversiones')

In [10]:
# el path al archivo de cd
df_secuencia = pd.read_csv(DATA_PATH / 'modelo_inversiones_completo_secuencia.csv',sep=';')
print(df_secuencia.loc[20]['codigo_completo'])

SELECT RF_PLI_044d_Modelo_Inversiones_Full.* INTO RF_PLI_Modelo_Inversiones_Final_CLP
FROM RF_PLI_044d_Modelo_Inversiones_Full;



In [11]:
'RF_PLI_Modelo_Inversiones_Final_CLP', #'RF_Tabla_Desarrollo_Final', 'RF_Tabla_Desarrollo_Interna'

('RF_PLI_Modelo_Inversiones_Final_CLP',)

---
# 🎯 PASOS 20-27: Tabla Final de Inversiones

A partir de aquí comenzamos con los pasos de salida.

## Paso 20: Generar Precios del Día (TCRC)

In [12]:
# Paso 20: Precios del día
df_precios_dia = generar_precios_dia(
    df_precios=tablas['RF_Base_Diaria_Precios'],
    fecha_proceso=FECHA_PROCESO,
    instrumento='TCRC',
    verbose=True
)

print(f"\n📊 Precios del día: {len(df_precios_dia)} registros")
display(df_precios_dia[df_precios_dia['NEMOTECNICO'].isin(['CLF', 'USD', 'CLP'])])


[Paso 20] Generando precios del día...
  Instrumento: TCRC
  ✓ Precios encontrados: 23 registros
    - CLF: 39,747.1200
    - USD: 883.0400

📊 Precios del día: 23 registros


,Fecha,NEMOTECNICO,Instrumento,Precio_Mid
454141,2026-01-15,CLF,TCRC,39747.12
454142,2026-01-15,CLP,TCRC,1.00
454155,2026-01-15,USD,TCRC,883.04


## Paso 21: Generar Tabla Final de Inversiones (UNION ALL)

In [13]:
# Paso 21: Tabla final de inversiones
df_tabla_final = generar_tabla_final_inversiones(
    flujos=flujos,
    fecha_proceso=FECHA_PROCESO,
    df_base=tablas['RF_base_Completa_Hist'],  # Para garantías
    df_cartera_inv_pacto=df_cartera_pacto,    # Pactos >90 días
    umbral_dias_pacto=90,
    verbose=True
)

print(f"\n📊 Tabla Final: {len(df_tabla_final):,} filas")
display(df_tabla_final.head(10))

[Paso 21] Generando tabla final de inversiones

  Procesando flujos de instrumentos:
    ✓ ML_C46_Inversiones_Financieras_GOBCLP: 6 flujos, total=623,701,831,416
    ✓ ML_C46_Inversiones_Financieras_GOBCLF: 2 flujos, total=1,084,688
    ✓ ML_C46_Inversiones_Financieras_DPFCLP: 1 flujos, total=25,992,461,906
    ✓ ML_C46_Inversiones_Financieras_DPRCLF: 5 flujos, total=523,605
    ✓ ML_C46_Inversiones_Financieras_CORPCLP: 64 flujos, total=108,127,396,636
    ✓ ML_C46_Inversiones_Financieras_CORPCLF: 4 flujos, total=1,569,972

  Generando cartera de garantías...
    ✓ Garantías: 11 registros, total=30,000,752,000

  Generando pactos fuera de plazo (>90 días)...
    ⚠ Sin pactos en cartera de entrada

  RESUMEN TABLA FINAL:
  - Total registros: 93
  - Total Cap_Amort: 787,825,620,223
  - Columnas: ['Fec_Pro', 'Cod_Emp', 'Moneda', 'Cod_A_P', 'Cod_Pro', 'Cod_Sub_Pro', 'Fec_Pago', 'Dias_Pago', 'Cap_Amort', 'Int_Total_Cont', 'VP_Cap_Amort', 'VP_Int_Total_Cont']

📊 Tabla Final: 93 filas


,Fec_Pro,Cod_Emp,Moneda,Cod_A_P,Cod_Pro,Cod_Sub_Pro,Fec_Pago,Dias_Pago,Cap_Amort,Int_Total_Cont,VP_Cap_Amort,VP_Int_Total_Cont
0,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-16,1.0,1.110155e+11,0.0,1.110155e+11,0.0
1,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-19,4.0,1.110155e+11,0.0,1.110155e+11,0.0
2,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-20,5.0,1.110155e+11,0.0,1.110155e+11,0.0
3,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-21,6.0,1.110155e+11,0.0,1.110155e+11,0.0
4,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-22,7.0,1.110155e+11,0.0,1.110155e+11,0.0
5,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLP,2026-01-23,8.0,6.862433e+10,0.0,6.862433e+10,0.0
6,2026-01-15,1,CLF,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLF,2026-01-16,1.0,7.369081e+05,0.0,7.369081e+05,0.0
7,2026-01-15,1,CLF,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_GOBCLF,2026-01-19,4.0,3.477801e+05,0.0,3.477801e+05,0.0
8,2026-01-15,1,CLP,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_DPFCLP,2026-01-16,1.0,2.599246e+10,0.0,2.599246e+10,0.0
9,2026-01-15,1,CLF,ACT,ML_C46_Inversiones_Financieras,ML_C46_Inversiones_Financieras_DPRCLF,2026-01-16,1.0,1.078761e+05,0.0,1.078761e+05,0.0


In [14]:
# comparamos fila por fila con tablas_access['RF_PLI_Modelo_Inversiones_Final_CLP']
if 'RF_PLI_Modelo_Inversiones_Final_CLP' in tablas_access:
    df_access = tablas_access['RF_PLI_Modelo_Inversiones_Final_CLP']
    print(f"\n📊 Comparando con Access: {len(df_access)} filas")
    
    # Asegurar mismo orden de columnas
    df_access = df_access[df_tabla_final.columns]
    
    # Partamos comparando totales, luego agrupados por Cod_Sub_Pro
    total_final = df_tabla_final['Cap_Amort'].sum()
    total_access = df_access['Cap_Amort'].sum()
    print(f"   - Total Tabla Final: {total_final:,.0f} CLP")
    print(f"   - Total Access: {total_access:,.0f} CLP")
    diferencia_total = total_final - total_access
    print(f"   - Diferencia Total: {diferencia_total:,.0f} CLP")

    # Comparar por Cod_Sub_Pro
    resumen_final = df_tabla_final.groupby('Cod_Sub_Pro')['Cap_Amort'].sum().reset_index()
    resumen_access = df_access.groupby('Cod_Sub_Pro')['Cap_Amort'].sum().reset_index()
    comparacion = resumen_final.merge(resumen_access, on='Cod_Sub_Pro', how='outer', suffixes=('_Final', '_Access')).fillna(0)
    comparacion['Diferencia'] = comparacion['Cap_Amort_Final'] - comparacion['Cap_Amort_Access']
    print("\n   - Comparación por Cod_Sub_Pro:")
    display(comparacion.sort_values('Diferencia', key=abs, ascending=False).head(10))


📊 Comparando con Access: 93 filas
   - Total Tabla Final: 787,825,620,223 CLP
   - Total Access: 787,825,620,223 CLP
   - Diferencia Total: 0 CLP

   - Comparación por Cod_Sub_Pro:


,Cod_Sub_Pro,Cap_Amort_Final,Cap_Amort_Access,Diferencia
0,ML_C46_Inversiones_Financieras_CORPCLF,1.569972e+06,1.569972e+06,-4.656613e-10
4,ML_C46_Inversiones_Financieras_GOBCLF,1.084688e+06,1.084688e+06,-2.328306e-10
1,ML_C46_Inversiones_Financieras_CORPCLP,1.081274e+11,1.081274e+11,0.000000e+00
2,ML_C46_Inversiones_Financieras_DPFCLP,2.599246e+10,2.599246e+10,0.000000e+00
3,ML_C46_Inversiones_Financieras_DPRCLF,5.236051e+05,5.236051e+05,0.000000e+00
5,ML_C46_Inversiones_Financieras_GOBCLP,6.237018e+11,6.237018e+11,0.000000e+00
6,ML_C46_Inversiones_Financieras_Gtia,3.000075e+10,3.000075e+10,0.000000e+00


In [ ]:
# Verificar distribución por Cod_Sub_Pro
print("\n📊 Distribución por Cod_Sub_Pro:")
resumen = df_tabla_final.groupby('Cod_Sub_Pro').agg({
    'Cap_Amort': ['count', 'sum']
}).round(0)
resumen.columns = ['Registros', 'Total_Cap_Amort']
display(resumen.sort_values('Total_Cap_Amort', ascending=False))

## Paso 23: Agregar Precio y Flujo CLP

In [ ]:
# Paso 23: Agregar precio UF y calcular Flujo_CLP
# Filtrar precio CLF (UF)
df_precio_clf = df_precios_dia[df_precios_dia['NEMOTECNICO'] == 'CLF']

df_con_precio = agregar_precio_y_flujo_clp(
    df_inversiones=df_tabla_final,
    df_precios_dia=df_precio_clf,
    verbose=True
)

print(f"\n📊 Tabla con precios: {len(df_con_precio):,} filas")
print(f"   Columnas: {list(df_con_precio.columns)}")

# Mostrar ejemplo de conversión CLF -> CLP
clf_rows = df_con_precio[df_con_precio['Moneda'] == 'CLF'].head(5)
if len(clf_rows) > 0:
    print("\n🔄 Ejemplo conversión CLF → CLP:")
    display(clf_rows[['Cod_Sub_Pro', 'Moneda', 'Cap_Amort', 'Precio_Mid', 'Flujo_CLP']])

## Pasos 24-26: Tabla Desarrollo Completa

In [ ]:
# Pasos 24-26: Generar tabla de desarrollo completa
# NOTA: FFMM, HTM, RT se extraen de cartera base si existen
df_tabla_desarrollo = generar_tabla_desarrollo_completa(
    df_modelo_inversiones=df_tabla_final,
    df_precios_dia=df_precio_clf,
    df_cartera_ffmm=None,  # Se puede pasar si hay FFMM
    df_cartera_htm=None,   # Se puede pasar si hay HTM
    df_cartera_rt=None,    # Se puede pasar si hay RT
    verbose=True
)

print(f"\n📊 Tabla Desarrollo: {len(df_tabla_desarrollo):,} filas")

## Paso 27: Formatear para Excel

In [ ]:
# Paso 27: Formatear para Excel
df_excel = formatear_para_excel(
    df_tabla_desarrollo=df_tabla_desarrollo,
    verbose=True
)

print(f"\n📊 Formato Excel: {len(df_excel):,} filas x {len(df_excel.columns)} columnas")
print(f"   Columnas: {list(df_excel.columns)}")
display(df_excel.head())

---
## 🧪 Validación vs Access

In [ ]:
# Comparar flujos generados vs Access
flujos_access = {}
for inst in ['GobCLP', 'GobCLF', 'DPF', 'DPR', 'BBC', 'LCH']:
    key = f'Flujo_{inst}'
    if key in tablas_access:
        flujos_access[inst] = tablas_access[key]

if flujos_access:
    print("="*70)
    print("COMPARACIÓN FLUJOS: Python vs Access")
    print("="*70)
    
    for inst in flujos.keys():
        if inst not in flujos_access:
            continue
            
        df_py = flujos[inst]
        df_acc = flujos_access[inst]
        
        # Comparar totales
        total_py = df_py['Monto_Liquidar'].sum() if 'Monto_Liquidar' in df_py.columns else 0
        total_acc = df_acc['Monto_Liquidar'].sum() if 'Monto_Liquidar' in df_acc.columns else 0
        diff = abs(total_py - total_acc)
        
        status = "✅" if diff < 1 else "⚠️"
        print(f"\n{inst}:")
        print(f"   Python: {total_py:20,.2f}")
        print(f"   Access: {total_acc:20,.2f}")
        print(f"   Diff:   {diff:20,.2f} {status}")
else:
    print("⚠️ No hay datos de Access disponibles para validación")

---
## 📦 Función Wrapper Completa

In [ ]:
# Ejecutar todo de una vez con la función wrapper
tablas_input = {
    'RF_Base_Diaria_Precios': tablas['RF_Base_Diaria_Precios'],
    'RF_base_Completa_Hist': tablas['RF_base_Completa_Hist']
}

resultado = ejecutar_pasos_20_a_27(
    flujos=flujos,
    tablas=tablas_input,
    fecha_proceso=FECHA_PROCESO,
    df_cartera_inv_pacto=df_cartera_pacto,
    verbose=True
)

print("\n" + "="*60)
print("📦 RESULTADO FINAL")
print("="*60)
print(f"✅ Precios día:      {len(resultado['precios_dia']):,} filas")
print(f"✅ Tabla final:      {len(resultado['tabla_final_inversiones']):,} filas")
print(f"✅ Tabla desarrollo: {len(resultado['tabla_desarrollo']):,} filas")
print(f"✅ Formato Excel:    {len(resultado['tabla_excel']):,} filas")

---
## 💾 Exportar a Excel (opcional)

In [ ]:
# Exportar a Excel si es necesario
EXPORTAR = False  # Cambiar a True para exportar

if EXPORTAR:
    output_path = DATA_PATH / f"ML_Inversiones_Output_{FECHA_PROCESO}.xlsx"
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        resultado['tabla_excel'].to_excel(writer, sheet_name='Tabla_Desarrollo', index=False)
        resultado['tabla_final_inversiones'].to_excel(writer, sheet_name='Tabla_Final', index=False)
    print(f"✅ Exportado a: {output_path}")
else:
    print("ℹ️ Exportación deshabilitada. Cambiar EXPORTAR=True para exportar.")